# Decision Procedure Ablation

Reproduces Table `tab:app-ablation` (Appendix, Sensitivity Analyses). Each metric in the decision tree is disabled in turn by overriding its value before classification, and all evaluations are re-classified.

**Setup:** baseline measurement regime, $N_{\text{eval}}=1000$, pure scenarios (S1, S2B, S3, S4h, S4r), pooled across signal strengths. Entangled = mean of CCA and CLIP. Factorized = mean of JIVE and DSSL, with DSSL averaged over its four $\beta$ configurations.

In [ ]:
import sys
from collections import defaultdict

import numpy as np
import pandas as pd

sys.path.insert(0, '../../experiments')
import ablation_study as A

from decimal import Decimal, ROUND_HALF_UP


def fmt(v: float, nd: int = 3) -> str:
    """Format with round-half-up, matching the paper (exact ties such as 0.5975 occur)."""
    return str(Decimal(f'{v:.10f}').quantize(Decimal(1).scaleb(-nd), rounding=ROUND_HALF_UP))

records = A.load_all_records(n_eval=1000, regime='baseline')
print(f'{len(records)} records, models: {sorted({r["model"] for r in records})}')

In [ ]:
SCENARIOS = ['scenario_1', 'scenario_2', 'scenario_3', 'scenario_4']
COLUMNS = ['S1', 'S2B', 'S3', 'S4', 'FSCR']


def accuracy_and_fscr(recs: list[dict], condition: str) -> np.ndarray:
    """Per-scenario strict accuracy and FSCR for one ablation condition.

    Parameters
    ----------
    recs : list of dict
        Result records for a single model.
    condition : str
        Ablation condition from ``A.ABLATION_CONDITIONS``.

    Returns
    -------
    np.ndarray
        Accuracy for S1, S2B, S3, S4 (NaN if absent), followed by FSCR.
    """
    correct, total = defaultdict(int), defaultdict(int)
    n_false_s1, n_non_s1 = 0, 0
    for r in recs:
        gt = A.get_ground_truth(r['alpha_config_id'])
        pred = A.classify_scenario(A.apply_ablation(A.build_metric_bundle(r), condition))
        total[gt] += 1
        correct[gt] += pred == gt
        if gt != 'scenario_1':
            n_non_s1 += 1
            n_false_s1 += pred == 'scenario_1'
    acc = [correct[s] / total[s] if total[s] else np.nan for s in SCENARIOS]
    return np.array(acc + [n_false_s1 / n_non_s1])


models = sorted({r['model'] for r in records})
rows = []
for cond in A.ABLATION_CONDITIONS:
    per_model = {m: accuracy_and_fscr([r for r in records if r['model'] == m], cond) for m in models}
    dssl = np.nanmean([v for m, v in per_model.items() if m.startswith('dssl')], axis=0)
    families = {
        'Entangled': np.mean([per_model['cca_K50'], per_model['clip_K50']], axis=0),
        'Factorized': np.mean([per_model['jive_K50'], dssl], axis=0),
    }
    for fam, vals in families.items():
        rows.append({'family': fam, 'condition': cond, **dict(zip(COLUMNS, vals))})

table = (pd.DataFrame(rows)
         .assign(condition=lambda d: pd.Categorical(d['condition'], A.ABLATION_CONDITIONS))
         .sort_values(['family', 'condition'])
         .set_index(['family', 'condition']))
print(table.to_string(float_format=fmt))